# Cross-Domain Transfer Learning Experiment
This notebook explores transfer learning between the ADKG and MDKG biomedical relation extraction datasets.

In [10]:
import os
import sys

## Environment Setup
Basic utilities for configuration management and experiment execution are imported.

In [17]:
import shutil

src = "../models/adkg_biobert/adkg_biobert/2026-05-05_21:05:50.256241/final_model/model.safetensors"

dst = "../models/adkg_biobert/adkg_biobert/2026-05-05_21:05:50.256241/final_model/pytorch_model.bin"

shutil.copy(src, dst)

print("Created pytorch_model.bin")

Created pytorch_model.bin


## Model Compatibility Preparation
This section prepares model files and compatibility adjustments required for older SpERT checkpoints.

In [22]:
from safetensors.torch import load_file
import torch

model_dir = "../models/adkg_biobert/adkg_biobert/2026-05-05_21:05:50.256241/final_model"

safe_path = f"{model_dir}/model.safetensors"
bin_path = f"{model_dir}/pytorch_model.bin"

# load safetensors
state_dict = load_file(safe_path)

# save true pytorch checkpoint
torch.save(state_dict, bin_path)

print("Converted to pytorch_model.bin")

Converted to pytorch_model.bin


## SafeTensor to PyTorch Conversion
The trained BioBERT + SpERT checkpoint is converted into a format compatible with the legacy evaluation pipeline.

In [23]:
adkg_model_path = "../models/adkg_biobert/adkg_biobert/2026-05-05_21:05:50.256241/final_model"

print(adkg_model_path)

../models/adkg_biobert/adkg_biobert/2026-05-05_21:05:50.256241/final_model


## Define Source Model Path
The pretrained ADKG model checkpoint is specified for transfer learning initialization.

In [28]:
config_text = f"""label = mdkg_transfer
model_type = spert
model_path = ../models/adkg_biobert/adkg_biobert/2026-05-05_21:05:50.256241/final_model
tokenizer_path = dmis-lab/biobert-base-cased-v1.1
train_path = ../data/spert_mdkg/train.json
valid_path = ../data/spert_mdkg/dev.json
types_path = ../data/spert_mdkg/types.json
train_batch_size = 1
eval_batch_size = 1
neg_entity_count = 100
neg_relation_count = 100
epochs = 5
lr = 5e-5
lr_warmup = 0.1
weight_decay = 0.01
max_grad_norm = 1.0
rel_filter_threshold = 0.4
size_embedding = 25
prop_drop = 0.1
max_span_size = 10
store_predictions = true
store_examples = true
sampling_processes = 1
max_pairs = 1000
final_eval = true
log_path = ../logs/mdkg_transfer
save_path = ../models/mdkg_transfer
"""

with open("../spert/configs/mdkg_transfer.conf", "w") as f:
    f.write(config_text)

## Create Transfer Learning Configuration
A new training configuration is generated to initialize MDKG training using pretrained ADKG representations.

In [29]:
command = f"{sys.executable} ../spert/spert.py train --config ../spert/configs/mdkg_transfer.conf"

os.system(command)

--------------------------------------------------
Config:
{'label': 'mdkg_transfer', 'model_type': 'spert', 'model_path': '../models/adkg_biobert/adkg_biobert/2026-05-05_21:05:50.256241/final_model', 'tokenizer_path': 'dmis-lab/biobert-base-cased-v1.1', 'train_path': '../data/spert_mdkg/train.json', 'valid_path': '../data/spert_mdkg/dev.json', 'types_path': '../data/spert_mdkg/types.json', 'train_batch_size': '1', 'eval_batch_size': '1', 'neg_entity_count': '100', 'neg_relation_count': '100', 'epochs': '5', 'lr': '5e-5', 'lr_warmup': '0.1', 'weight_decay': '0.01', 'max_grad_norm': '1.0', 'rel_filter_threshold': '0.4', 'size_embedding': '25', 'prop_drop': '0.1', 'max_span_size': '10', 'store_predictions': 'true', 'store_examples': 'true', 'sampling_processes': '1', 'max_pairs': '1000', 'final_eval': 'true', 'log_path': '../logs/mdkg_transfer', 'save_path': '../models/mdkg_transfer'}
Repeat 1 times
--------------------------------------------------
Iteration 0
--------------------------

Parse dataset 'valid': 100%|██████████| 941/941 [00:02<00:00, 432.94it/s]


2026-05-06 10:06:39,312 [MainThread  ] [INFO ]  Relation type count: 10
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  Entity type count: 10
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  Entities:
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  No Entity=0
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  Health_factors=1
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  disease=2
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  drug=3
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  gene=4
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  method=5
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  physiology=6
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  region=7
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  signs=8
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  symptom=9
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  Relations:
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  No Relation=0
2026-05-06 10:06:39,313 [MainThread  ] [INFO ]  abbreviation_for=1
2026-05-06 10:06:39,313 [MainT

Process SpawnProcess-1:
Traceback (most recent call last):
  File "/usr/lib64/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib64/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/nas/longleaf/home/ahabib/final/spert/spert.py", line 16, in __train
    trainer.train(train_path=run_args.train_path, valid_path=run_args.valid_path,
  File "/nas/longleaf/home/ahabib/final/spert/spert/spert_trainer.py", line 65, in train
    model = self._load_model(input_reader)
  File "/nas/longleaf/home/ahabib/final/spert/spert/spert_trainer.py", line 158, in _load_model
    model = model_class.from_pretrained(self._args.model_path,
  File "/nas/longleaf/home/ahabib/final/venv/lib64/python3.9/site-packages/transformers/modeling_utils.py", line 277, in _wrapper
    return func(*args, **kwargs)
  File "/nas/longleaf/home/ahabib/final/venv/lib64/python3.9/site-packages/transformers/modeling_utils.py", li

0

## Launch Cross-Domain Fine-Tuning
The following command starts the transfer-learning experiment.